In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

-

### Evaluate the tuned best HOG + SGD linear SVM on the test set

> authored by Han Qin z5565186

In [ ]:
from pathlib import Path
import json
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------
project_dir = Path("/content/drive/MyDrive/COMP9517_Group_Assignment")

feature_dir = project_dir / "data" / "hog_features"
tuning_result_dir = project_dir / "results" / "tuning_result"

# The final best model saved by the tuning notebook.
best_model_file = tuning_result_dir / "best_hog_sgd_svm.pkl"

# Test features and labels created by the feature-extraction stage.
X_test_file = feature_dir / "X_test.npy"
y_test_file = feature_dir / "y_test.npy"

# Evaluation outputs.
test_result_dir = tuning_result_dir / "test_evaluation"
test_result_dir.mkdir(parents=True, exist_ok=True)

summary_file = test_result_dir / "test_summary_metrics.csv"
per_class_file = test_result_dir / "test_per_class_metrics.csv"
report_file = test_result_dir / "test_classification_report.csv"
confusion_matrix_file = test_result_dir / "test_confusion_matrix.csv"
normalised_confusion_matrix_file = (
    test_result_dir / "test_confusion_matrix_normalised.csv"
)
confusion_matrix_figure_file = (
    test_result_dir / "test_confusion_matrix.png"
)
normalised_confusion_matrix_figure_file = (
    test_result_dir / "test_confusion_matrix_normalised.png"
)
predictions_file = test_result_dir / "test_predictions.npy"
model_parameters_file = test_result_dir / "best_model_parameters.json"


# ------------------------------------------------------------
# 2. Validate and load files
# ------------------------------------------------------------
required_files = [best_model_file, X_test_file, y_test_file]

missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

with open(best_model_file, "rb") as file:
    best_model = pickle.load(file)

X_test = np.load(X_test_file)
y_test = np.load(y_test_file)

if len(X_test) != len(y_test):
    raise ValueError(
        f"X_test and y_test have different sample counts: "
        f"{len(X_test)} and {len(y_test)}"
    )

if X_test.ndim != 2:
    raise ValueError(
        f"Expected X_test to be a 2-D feature matrix, got shape {X_test.shape}"
    )

print("Loaded best model from:")
print(best_model_file)
print()
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


# ------------------------------------------------------------
# 3. Run inference
# ------------------------------------------------------------
start_time = time.perf_counter()
y_pred = best_model.predict(X_test)
prediction_time = time.perf_counter() - start_time

np.save(predictions_file, y_pred)

# Use the model's complete class order so rows and columns remain stable,
# even if a class happens to be absent from y_test or y_pred.
labels = np.asarray(best_model.classes_)


# ------------------------------------------------------------
# 4. Overall metrics
# ------------------------------------------------------------
test_accuracy = accuracy_score(y_test, y_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_pred)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0,
    )
)

micro_precision, micro_recall, micro_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=labels,
        average="micro",
        zero_division=0,
    )
)

summary_metrics = {
    "model": type(best_model).__name__,
    "loss": getattr(best_model, "loss", None),
    "penalty": getattr(best_model, "penalty", None),
    "alpha": getattr(best_model, "alpha", None),
    "average": getattr(best_model, "average", None),
    "max_iter": getattr(best_model, "max_iter", None),
    "tolerance": getattr(best_model, "tol", None),
    "random_state": getattr(best_model, "random_state", None),
    "iterations": int(np.max(best_model.n_iter_)),
    "number_of_test_samples": int(len(y_test)),
    "number_of_features": int(X_test.shape[1]),
    "number_of_classes": int(len(labels)),
    "test_accuracy": test_accuracy,
    "test_balanced_accuracy": test_balanced_accuracy,
    "test_macro_precision": macro_precision,
    "test_macro_recall": macro_recall,
    "test_macro_f1": macro_f1,
    "test_weighted_precision": weighted_precision,
    "test_weighted_recall": weighted_recall,
    "test_weighted_f1": weighted_f1,
    "test_micro_precision": micro_precision,
    "test_micro_recall": micro_recall,
    "test_micro_f1": micro_f1,
    "total_prediction_time_seconds": prediction_time,
    "prediction_time_per_sample_seconds": prediction_time / len(y_test),
}

summary_df = pd.DataFrame([summary_metrics])
summary_df.to_csv(summary_file, index=False)


# ------------------------------------------------------------
# 5. Per-class precision, recall, F1-score and support
# ------------------------------------------------------------
class_precision, class_recall, class_f1, class_support = (
    precision_recall_fscore_support(
        y_test,
        y_pred,
        labels=labels,
        average=None,
        zero_division=0,
    )
)

per_class_df = pd.DataFrame(
    {
        "class": labels,
        "precision": class_precision,
        "recall": class_recall,
        "f1_score": class_f1,
        "support": class_support.astype(int),
    }
)

per_class_df.to_csv(per_class_file, index=False)


# ------------------------------------------------------------
# 6. Complete sklearn classification report
# ------------------------------------------------------------
report_dict = classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=[str(label) for label in labels],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(report_file)



# ------------------------------------------------------------
# 7. Save the selected model parameters
# ------------------------------------------------------------
model_parameters = best_model.get_params(deep=True)

# Convert non-JSON-native values safely.
serialisable_parameters = {
    key: (
        value.item()
        if isinstance(value, np.generic)
        else value
    )
    for key, value in model_parameters.items()
}

with open(model_parameters_file, "w", encoding="utf-8") as file:
    json.dump(
        serialisable_parameters,
        file,
        indent=4,
        default=str,
    )



Loaded best model from:
/content/drive/MyDrive/COMP9517_Group_Assignment/results/tuning_result/best_hog_sgd_svm.pkl

X_test shape: (5000, 8100)
y_test shape: (5000,)


### Display result

In [ ]:
print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Accuracy:             {test_accuracy:.4f}")
print(f"Macro Precision:      {macro_precision:.4f}")
print(f"Macro Recall:         {macro_recall:.4f}")
print(f"Macro F1-score:       {macro_f1:.4f}")
print(f"Inference Time:       {prediction_time:.4f} seconds")


FINAL TEST RESULTS
Accuracy:             0.0198
Macro Precision:      0.0259
Macro Recall:         0.0198
Macro F1-score:       0.0198
Inference Time:       0.3767
